#                                   DLA Exp8 -AutoEncoder

In [1]:
import numpy as np
import tensorflow as tf 
from tensorflow.keras.datasets import imdb

# Set seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Parameters
VOCAB_SIZE = 5000  # Retain top 5,000 most frequent words


In [ ]:
# Load raw integer sequences from Keras IMDB dataset
(train_data, y_train), (test_data, y_test) = imdb.load_data(num_words=VOCAB_SIZE)
# Fetch word index mapping to reconstruct original text strings
word_index = imdb.get_word_index()

# IMDB dataset shifts indices by 3 to reserve 0, 1, 2, 3 for special tokens
reverse_word_index = {value + 3: key for key, value in word_index.items()}
reverse_word_index[0] = "<PAD>"
reverse_word_index[1] = "<START>"
reverse_word_index[2] = "<UNK>"
reverse_word_index[3] = "<UNUSED>"

def decode_review(text_ids):
    """Converts a list of integer IDs back into a space-separated string."""
    return " ".join([reverse_word_index.get(i, "?") for i in text_ids])

# Reconstruct raw review strings
X_train_text = [decode_review(review) for review in train_data]
X_test_text  = [decode_review(review) for review in test_data]

print(f"Loaded {len(X_train_text)} training reviews and {len(X_test_text)} test reviews.")
print("\nSample Decoded Review:\n", X_train_text[0][:200], "...")

In [ ]:
# ============================================================
#TF-IDF Vectorization
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer constrained to top 5,000 terms
# sublinear_tf=True applies log(1+tf) scaling which dampens the effect of very high frequency terms
# ngram_range=(1,2) adds bigrams to capture simple phrases like 'not good', 'very bad'
tfidf = TfidfVectorizer(max_features=VOCAB_SIZE, stop_words='english',
                        sublinear_tf=True, ngram_range=(1, 2))

# Fit on training text and transform both train and test sets
X_train = tfidf.fit_transform(X_train_text).toarray().astype('float32')
X_test  = tfidf.transform(X_test_text).toarray().astype('float32')

print(f"X_train shape: {X_train.shape}")  # Expected: (25000, 5000)
print(f"X_test shape : {X_test.shape}")   # Expected: (25000, 5000)
print(f"Sparsity check - Non-zero entries in sample 0: {np.count_nonzero(X_train[0])} / 5000")

In [ ]:
# ============================================================
# Deep Autoencoder Architecture & Training
# ============================================================
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

# Matrix dimension (5,000 TF-IDF features)
input_dim = X_train.shape[1]  

# Define Model Architecture
# 5000 -> 512 -> 128 -> 64 (Latent Bottleneck) -> 128 -> 512 -> 5000
input_layer = Input(shape=(input_dim,), name='tfidf_input')

# --- ENCODER ---
# Progressively compresses 5,000 sparse features down to 64 core latent dimensions
encoded = Dense(512, activation='relu', name='encoder_1')(input_layer)
encoded = BatchNormalization()(encoded)
encoded = Dropout(0.3)(encoded)  # Prevents memorization of common words
encoded = Dense(128, activation='relu', name='encoder_2')(encoded)
encoded = BatchNormalization()(encoded)
encoded = Dropout(0.2)(encoded)
latent  = Dense(64, activation='relu', name='latent_bottleneck')(encoded)

# --- DECODER ---
# Reconstructs the 5,000 TF-IDF values from the 64-dimensional bottleneck
decoded = Dense(128, activation='relu', name='decoder_1')(latent)
decoded = BatchNormalization()(decoded)
decoded = Dense(512, activation='relu', name='decoder_2')(decoded)
decoded = Dense(input_dim, activation='sigmoid', name='reconstruction_output')(decoded)

# Build & Compile Autoencoder
autoencoder = Model(inputs=input_layer, outputs=decoded, name='IMDB_Autoencoder')

# Binary Cross-Entropy works best here because TF-IDF values are scaled [0, 1]
autoencoder.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='binary_crossentropy')

autoencoder.summary()

# --- TRAIN THE AUTOENCODER ---
# Target is X_train itself (Unsupervised Learning)
# EarlyStopping restores the best weights to avoid overfitting
early_stop_ae = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("\nTraining Autoencoder...")
history = autoencoder.fit(
    X_train, X_train,
    epochs=30,
    batch_size=256,
    validation_split=0.2,
    callbacks=[early_stop_ae],
    verbose=1
)

In [ ]:
# ============================================================
#  Extract Bottleneck Features via Encoder
# ============================================================
from tensorflow.keras.models import Model

# Build an isolated Encoder model mapping input -> latent bottleneck layer
encoder = Model(
    inputs=autoencoder.input, 
    outputs=autoencoder.get_layer('latent_bottleneck').output, 
    name='Encoder_Only'
)

# Predict / transform high-dim inputs to 64-dim latent vectors
X_train_encoded = encoder.predict(X_train, batch_size=256)
X_test_encoded  = encoder.predict(X_test, batch_size=256)

print(f"Original X_train shape : {X_train.shape}")       # (25000, 5000)
print(f"Encoded X_train shape  : {X_train_encoded.shape}") # (25000, 64)
print("\nSample 64-dimensional bottleneck vector for Review #0:\n", X_train_encoded[0])

In [ ]:
# ============================================================
# Downstream Sentiment Classifier
# ============================================================
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, accuracy_score

# Build a deeper MLP for classification on 64-dim latent features
classifier = Sequential([
    Dense(64, activation='relu', input_shape=(64,)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(1, activation='sigmoid')  # Output probability: >0.5 = Positive, <=0.5 = Negative
], name='Sentiment_Classifier')

# Compile with Binary Cross-Entropy
classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Callbacks: reduce LR when validation loss plateaus, stop early if no improvement
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
early_stop_cls = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

# Train using latent features (X_train_encoded) + sentiment labels (y_train)
print("Training Supervised Classifier on 64-dim features...")
history_cls = classifier.fit(
    X_train_encoded, y_train,
    epochs=40,
    batch_size=128,
    validation_split=0.2,
    callbacks=[reduce_lr, early_stop_cls],
    verbose=1
)

# Evaluate on held-out test set
y_pred_prob = classifier.predict(X_test_encoded)
y_pred = (y_pred_prob > 0.5).astype(int).ravel()

print("\n" + "="*50)
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("="*50)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

In [ ]:
# ============================================================
# Model Performance & Latent Space Visualization
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE

# Set plot styling
sns.set_theme(style='whitegrid')
plt.figure(figsize=(18, 5))

# ------------------------------------------------------------
# PLOT 1: Autoencoder Reconstruction Loss
# ------------------------------------------------------------
plt.subplot(1, 3, 1)
plt.plot(history.history['loss'], label='Train Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Val Loss', linewidth=2, linestyle='--')
plt.title('Autoencoder Reconstruction Loss', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Binary Cross-Entropy Loss')
plt.legend()

# ------------------------------------------------------------
# PLOT 2: Classifier Accuracy & Loss
# ------------------------------------------------------------
plt.subplot(1, 3, 2)
plt.plot(history_cls.history['accuracy'], label='Train Acc', linewidth=2)
plt.plot(history_cls.history['val_accuracy'], label='Val Acc', linewidth=2, linestyle='--')
plt.title('Classifier Sentiment Accuracy', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# ------------------------------------------------------------
# PLOT 3: 2D t-SNE Projection of Latent Bottleneck (64D -> 2D)
# ------------------------------------------------------------
plt.subplot(1, 3, 3)
print("Computing t-SNE projection on 2,000 test samples...")

# Sample 2,000 test points for faster t-SNE computation
sample_idx = np.random.choice(len(X_test_encoded), size=2000, replace=False)
X_sample = X_test_encoded[sample_idx]
y_sample = y_test[sample_idx]

# Reduce 64D latent vector to 2D
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_sample)

# Scatter plot colored by ground-truth sentiment
scatter = plt.scatter(
    X_tsne[:, 0], X_tsne[:, 1], 
    c=y_sample, 
    cmap='coolwarm', 
    alpha=0.6, 
    s=15
)
plt.colorbar(scatter, ticks=[0, 1], label='Sentiment (0 = Neg, 1 = Pos)')
plt.title('t-SNE Latent Space Clusters (64D → 2D)', fontsize=12, fontweight='bold')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')

plt.tight_layout()
plt.show()